# 실습 02 · Token·ID·Context 분석 (기본필수) — 단계별 노트북

`main.py`의 흐름을 **셀 하나에 한 가지 확인**씩 잘게 나눴습니다. 위에서부터 순서대로 실행하세요.

| 단계 | 하는 일 |
|---|---|
| 0 | 준비: 경로 설정, 데이터 보기 |
| 1 | Tokenizer(단어 사전) 만들기 |
| B01 | 문장 A/B/C → Token·ID, 개수 세기 |
| B02 | 길이 맞추기(Padding)와 Mask |
| 실험 | 잘림(Truncation), 모르는 단어(UNK) |
| B03 | Context 예산 계산 |
| B04 | 결과 저장 및 확인 |

> 각 셀의 **마지막 줄에 변수 이름만 쓰면** 값이 바로 아래에 표시됩니다.

## 0. 준비

### 0-1. 경로 설정과 import
Jupyter에는 `__file__`이 없으므로 노트북이 실행되는 폴더(`02_tokenization`)를 기준으로 삼습니다.

In [2]:
from pathlib import Path
import sys
import json

HERE = Path.cwd()
if not (HERE / "main.py").exists():
    raise RuntimeError(f"노트북 실행 폴더가 02_tokenization이 아닙니다: {HERE}")
LEVEL_ROOT = HERE.parent  # 01_basic 폴더 (common 패키지가 있는 곳)
if str(LEVEL_ROOT) not in sys.path:
    sys.path.insert(0, str(LEVEL_ROOT))  # 셀을 여러 번 실행해도 중복 추가하지 않습니다.

from common.support import DATA, ToyTokenizer, context_budget, write_json, words

print("HERE       :", HERE)
print("LEVEL_ROOT :", LEVEL_ROOT)

HERE       : c:\workspace\practice_assignment\20261113\mentor\01_basic\02_tokenization
LEVEL_ROOT : c:\workspace\practice_assignment\20261113\mentor\01_basic


In [3]:
# 표 형태로 보기 좋게 출력하는 작은 도우미 함수
def show(rows, cols):
    widths = [max(len(str(c)), *(len(str(r[c])) for r in rows)) for c in cols]
    print(" | ".join(str(c).ljust(w) for c, w in zip(cols, widths)))
    print("-+-".join("-" * w for w in widths))
    for r in rows:
        print(" | ".join(str(r[c]).ljust(w) for c, w in zip(cols, widths)))

### 0-2. 실습 데이터 보기
`DATA`는 지식 문장 3건(A, B, C)입니다. 11/12에 DB에 넣었던 지식 데이터와 같은 형태입니다.

In [4]:
DATA

[{'id': 'A',
  'title': '실습 파일 보관',
  'content': '실습 파일은 개인 폴더에 보관합니다.',
  'category': 'learning'},
 {'id': 'B',
  'title': '서버 실행',
  'content': '프로젝트 폴더에서 서버를 실행합니다.',
  'category': 'environment'},
 {'id': 'C',
  'title': '질문 작성',
  'content': '오류 메시지와 시도한 작업을 함께 적습니다.',
  'category': 'learning'}]

In [5]:
show(DATA, ["id", "title", "content"])

id | title    | content                 
---+----------+-------------------------
A  | 실습 파일 보관 | 실습 파일은 개인 폴더에 보관합니다.    
B  | 서버 실행    | 프로젝트 폴더에서 서버를 실행합니다.    
C  | 질문 작성    | 오류 메시지와 시도한 작업을 함께 적습니다.


### 0-3. 문장만 뽑기
Tokenizer에 넣을 것은 `content`(실제 문장)뿐입니다.

In [6]:
texts = [item["content"] for item in DATA]
texts

['실습 파일은 개인 폴더에 보관합니다.', '프로젝트 폴더에서 서버를 실행합니다.', '오류 메시지와 시도한 작업을 함께 적습니다.']

## 1. Tokenizer(단어 사전) 만들기

### 1-1. 문장을 단어 조각으로 자르기 — `words()`
ToyTokenizer는 내부에서 `words()`로 문장을 자릅니다. 한글 덩어리 / 영어 / 숫자 / 문장부호 단위로 나눕니다.

In [7]:
words(texts[0])

['실습', '파일은', '개인', '폴더에', '보관합니다', '.']

In [8]:
# 영어·한글·기호가 섞이면?
words("FastAPI와 PostgreSQL"), words("print(x) // code")

(['FastAPI', '와', 'PostgreSQL'], ['print', '(', 'x', ')', '/', '/', 'code'])

### 1-2. 사전 만들기
A/B/C 문장 + 실습에 쓸 추가 문장 3개에서 나온 단어로 사전을 만듭니다.

In [9]:
tok = ToyTokenizer(texts + ["AI", "FastAPI와 PostgreSQL", "print(x) // code"])
len(tok.vocab)

30

### 1-3. 사전 내용 보기
앞의 0~3번은 **특수 Token**입니다.

| Token | 의미 |
|---|---|
| `[PAD]` | 길이 맞추기용 빈칸 |
| `[UNK]` | 사전에 없는 단어 |
| `[BOS]` | 문장 시작 |
| `[EOS]` | 문장 끝 |

In [10]:
tok.vocab

{'[PAD]': 0,
 '[UNK]': 1,
 '[BOS]': 2,
 '[EOS]': 3,
 '(': 4,
 ')': 5,
 '.': 6,
 '/': 7,
 'AI': 8,
 'FastAPI': 9,
 'PostgreSQL': 10,
 'code': 11,
 'print': 12,
 'x': 13,
 '개인': 14,
 '메시지와': 15,
 '보관합니다': 16,
 '서버를': 17,
 '시도한': 18,
 '실습': 19,
 '실행합니다': 20,
 '오류': 21,
 '와': 22,
 '작업을': 23,
 '적습니다': 24,
 '파일은': 25,
 '폴더에': 26,
 '폴더에서': 27,
 '프로젝트': 28,
 '함께': 29}

In [11]:
# 반대 방향 사전: 번호 → 단어
tok.reverse[19], tok.reverse[6], tok.reverse[2]

('실습', '.', '[BOS]')

## B01. 문장 A/B/C → Token·ID

먼저 **문장 A 하나만** 자세히 본 다음, 반복문으로 A/B/C를 처리합니다.

### B01-1. 문장 A를 encode

In [12]:
item = DATA[0]
item

{'id': 'A',
 'title': '실습 파일 보관',
 'content': '실습 파일은 개인 폴더에 보관합니다.',
 'category': 'learning'}

In [13]:
record = tok.encode(item["content"])
record

{'tokens': ['[BOS]', '실습', '파일은', '개인', '폴더에', '보관합니다', '.', '[EOS]'],
 'input_ids': [2, 19, 25, 14, 26, 16, 6, 3],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1],
 'valid_length': 8,
 'truncated': False,
 'original_length': 8}

### B01-2. record 안에 무엇이 들어 있나

In [14]:
list(record.keys())

['tokens',
 'input_ids',
 'attention_mask',
 'valid_length',
 'truncated',
 'original_length']

In [15]:
record["tokens"]       # 사람이 읽는 조각

['[BOS]', '실습', '파일은', '개인', '폴더에', '보관합니다', '.', '[EOS]']

In [16]:
record["input_ids"]    # LLM에 실제로 들어가는 숫자

[2, 19, 25, 14, 26, 16, 6, 3]

In [17]:
# Token과 ID를 짝지어 보기
list(zip(record["tokens"], record["input_ids"]))

[('[BOS]', 2),
 ('실습', 19),
 ('파일은', 25),
 ('개인', 14),
 ('폴더에', 26),
 ('보관합니다', 16),
 ('.', 6),
 ('[EOS]', 3)]

In [18]:
record["attention_mask"]   # 1 = 진짜 Token, 0 = PAD (지금은 PAD가 없어서 전부 1)

[1, 1, 1, 1, 1, 1, 1, 1]

### B01-3. Token 개수 세기
보이는 단어는 6개인데 Token 수는 8개입니다. `[BOS]`, `[EOS]` 2개가 더해지기 때문입니다.

In [19]:
print("단어 수 (words)        :", len(words(item["content"])))
print("Token 수 (valid_length):", record["valid_length"])
print("특수 Token 수          :", record["valid_length"] - len(words(item["content"])))

단어 수 (words)        : 6
Token 수 (valid_length): 8
특수 Token 수          : 2


### B01-4. A/B/C 전체를 반복문으로 처리 (`main.py`와 같은 코드)

In [20]:
rows = []
for item in DATA:
    record = tok.encode(item["content"])
    rows.append({"id": item["id"], "text": item["content"],
                 "tokens": record["tokens"], "ids": record["input_ids"],
                 "count": record["valid_length"], "special_count": 2})
len(rows)

3

In [21]:
show(rows, ["id", "count", "ids"])

id | count | ids                              
---+-------+----------------------------------
A  | 8     | [2, 19, 25, 14, 26, 16, 6, 3]    
B  | 7     | [2, 28, 27, 17, 20, 6, 3]        
C  | 9     | [2, 21, 15, 18, 23, 29, 24, 6, 3]


In [22]:
for r in rows:
    print(r["id"], r["tokens"])

A ['[BOS]', '실습', '파일은', '개인', '폴더에', '보관합니다', '.', '[EOS]']
B ['[BOS]', '프로젝트', '폴더에서', '서버를', '실행합니다', '.', '[EOS]']
C ['[BOS]', '오류', '메시지와', '시도한', '작업을', '함께', '적습니다', '.', '[EOS]']


### B01-5. 확인
예상 결과: (A, B, C) = (8, 7, 9)

In [23]:
counts = [(r["id"], r["count"]) for r in rows]
assert counts == [("A", 8), ("B", 7), ("C", 9)], counts
print("B01 통과 ✅", counts)

B01 통과 ✅ [('A', 8), ('B', 7), ('C', 9)]


In [24]:
# 관찰: 같은 Token은 항상 같은 ID (문장부호 '.' = 6)
# 관찰: '폴더에'와 '폴더에서'는 뜻이 비슷해도 다른 Token
tok.vocab["."], tok.vocab["폴더에"], tok.vocab["폴더에서"]

(6, 26, 27)

## B02. 길이 맞추기(Padding)와 Mask

LLM은 여러 문장을 한 번에 처리할 때 **모든 문장의 길이가 같아야** 합니다.

### B02-1. 따로 encode하면 길이가 다르다

In [25]:
a = tok.encode("AI")
b = tok.encode("FastAPI와 PostgreSQL")
len(a["input_ids"]), len(b["input_ids"])

(3, 5)

In [26]:
a["tokens"], b["tokens"]

(['[BOS]', 'AI', '[EOS]'], ['[BOS]', 'FastAPI', '와', 'PostgreSQL', '[EOS]'])

### B02-2. `batch()`로 한 번에 처리하면 길이가 맞춰진다
짧은 쪽에 `[PAD]`(ID 0)가 채워집니다.

In [27]:
batch = tok.batch(["AI", "FastAPI와 PostgreSQL"])
len(batch)

2

In [28]:
for x in batch:
    print("tokens:", x["tokens"])
    print("ids   :", x["input_ids"])
    print("mask  :", x["attention_mask"])
    print()

tokens: ['[BOS]', 'AI', '[EOS]', '[PAD]', '[PAD]']
ids   : [2, 8, 3, 0, 0]
mask  : [1, 1, 1, 0, 0]

tokens: ['[BOS]', 'FastAPI', '와', 'PostgreSQL', '[EOS]']
ids   : [2, 9, 22, 10, 3]
mask  : [1, 1, 1, 1, 1]



### B02-3. Padding 길이 vs 실제 Token 수
**가장 헷갈리는 부분입니다.** PAD를 채운 뒤에도 실제 Token 수(`valid_length`)는 그대로입니다.

In [29]:
for x in batch:
    print(f"전체 길이 {len(x['input_ids'])}  /  실제 Token {x['valid_length']}  /  PAD {x['attention_mask'].count(0)}")

전체 길이 5  /  실제 Token 3  /  PAD 2
전체 길이 5  /  실제 Token 5  /  PAD 0


In [30]:
# Mask가 0인 위치 = PAD 위치
[(t, m) for t, m in zip(batch[0]["tokens"], batch[0]["attention_mask"])]

[('[BOS]', 1), ('AI', 1), ('[EOS]', 1), ('[PAD]', 0), ('[PAD]', 0)]

### B02-4. 확인 (`main.py`의 assert 2개)

In [31]:
assert len(batch[0]["input_ids"]) == len(batch[1]["input_ids"])
assert all(len(x["input_ids"]) == len(x["attention_mask"]) for x in batch)
print("B02 통과 ✅ 두 문장 길이 =", len(batch[0]["input_ids"]))

B02 통과 ✅ 두 문장 길이 = 5


## 실험. 잘림(Truncation)과 모르는 단어(UNK)
과제에는 없지만, 실제 Tokenizer에서도 똑같이 일어나는 현상입니다. 숫자를 바꿔 가며 실행해 보세요.

### 실험-1. `max_length`로 자르기

In [32]:
cut = tok.encode(texts[0], max_length=5)   # 5를 바꿔 보세요 (2 이상)
print("원래 길이:", cut["original_length"])
print("잘린 결과:", cut["tokens"])
print("잘렸나?  :", cut["truncated"])

원래 길이: 8
잘린 결과: ['[BOS]', '실습', '파일은', '개인', '[EOS]']
잘렸나?  : True


### 실험-2. 사전에 없는 단어 → `[UNK]`(ID 1)

In [33]:
unk = tok.encode("새로운 단어 실습")   # 아무 문장이나 넣어 보세요
list(zip(unk["tokens"], unk["input_ids"]))

[('[BOS]', 2), ('새로운', 1), ('단어', 1), ('실습', 19), ('[EOS]', 3)]

## B03. Context 예산 계산

LLM이 한 번에 다룰 수 있는 Token 한도(Context Window) 안에 **입력 + 답변 공간**이 모두 들어가야 합니다.

### B03-1. 입력 구성

In [34]:
window = 5000
parts = {"system": 300, "history": 1200, "evidence": 2000, "user": 150}
output_reserved = 850
parts

{'system': 300, 'history': 1200, 'evidence': 2000, 'user': 150}

In [35]:
input_used = sum(parts.values())
input_used

3650

In [36]:
# 직접 계산
print("입력 합계 :", input_used)
print("출력 예약 :", output_reserved)
print("총 사용   :", input_used + output_reserved)
print("남는 공간 :", window - input_used - output_reserved)

입력 합계 : 3650
출력 예약 : 850
총 사용   : 4500
남는 공간 : 500


### B03-2. `context_budget()` 함수로 계산 (`main.py`와 같은 코드)

In [37]:
budget = context_budget(window, parts, output_reserved)
budget

{'window': 5000,
 'input_used': 3650,
 'output_reserved': 850,
 'remaining': 500,
 'fits': True}

In [38]:
assert budget["remaining"] == 500 and budget["fits"]
print("B03 통과 ✅ 남는 공간 =", budget["remaining"])

B03 통과 ✅ 남는 공간 = 500


### B03-3. 실험: 근거 문서(evidence)를 늘리면?

In [39]:
bigger = dict(parts, evidence=2600)   # 숫자를 바꿔 보세요
context_budget(window, bigger, output_reserved)

{'window': 5000,
 'input_used': 4250,
 'output_reserved': 850,
 'remaining': -100,
 'fits': False}

## B04. 결과 저장

### B04-1. 결과 모으기

In [40]:
trunc_cases = {}       # 응용·심화용 자리 (기본필수는 빈 값)
batch_report = []
mixed_rows = []
small_budget = None
policy = "예산과 잘림의 차이를 설명"

result = {"tokenizer_kind": "교육용 regex·고정 어휘표 (사전학습 모델이 아님)",
          "records": rows, "batch": batch, "budget": budget, "truncation": trunc_cases,
          "batch_report": batch_report, "language_comparison": mixed_rows,
          "overflow_budget": small_budget, "policy": policy}
list(result.keys())

['tokenizer_kind',
 'records',
 'batch',
 'budget',
 'truncation',
 'batch_report',
 'language_comparison',
 'overflow_budget',
 'policy']

### B04-2. 파일로 저장

In [41]:
out_path = HERE / "outputs" / "token_report.json"
write_json(out_path, result)
out_path

WindowsPath('c:/workspace/practice_assignment/20261113/mentor/01_basic/02_tokenization/outputs/token_report.json')

### B04-3. 저장된 파일 다시 읽어서 확인

In [42]:
saved = json.loads(out_path.read_text(encoding="utf-8"))
saved["budget"]

{'window': 5000,
 'input_used': 3650,
 'output_reserved': 850,
 'remaining': 500,
 'fits': True}

In [43]:
[(r["id"], r["count"]) for r in saved["records"]]

[('A', 8), ('B', 7), ('C', 9)]

### B04-4. `main.py`와 같은 최종 출력

In [44]:
print("[02]", [(x["id"], x["count"]) for x in rows], "remaining", budget["remaining"])

[02] [('A', 8), ('B', 7), ('C', 9)] remaining 500


---
### 정리
- **B01**: 문장 → Token → ID. Token 수에는 `[BOS]`/`[EOS]`가 포함된다.
- **B02**: 여러 문장을 묶으면 PAD로 길이를 맞추고, Mask 0이 PAD 위치다. 실제 Token 수는 `valid_length`.
- **B03**: Context Window에는 입력뿐 아니라 **답변 공간**도 남겨야 한다.
- **B04**: 결과는 `outputs/token_report.json`에 저장된다.